In [1]:
import pandas as pd
import numpy as np

schools = pd.read_csv("../data/processed/schools_clean.csv")        # our 3,147-school universe
inst = pd.read_excel("../data/raw/eada/instLevel.xlsx")             # EADA: one row per school
sports = pd.read_excel("../data/raw/eada/schools.xlsx")             # EADA: one row per school x sport

# Clean hidden whitespace in text columns (e.g. "Other " vs "Other") so exact matches work
for frame in (inst, sports):
    for col in ["classification_name", "ClassificationOther", "Sports", "institution_name"]:
        if col in frame.columns:
            frame[col] = frame[col].astype("string").str.strip()
print("Scorecard:", schools.shape, "| EADA schools:", inst.shape, "| EADA sport rows:", sports.shape)

# Confirm every EADA column we rely on exists before using it
needed_inst = ["unitid", "classification_name", "STUDENTAID_MEN", "STUDENTAID_WOMEN", "STUDENTAID_TOTAL",
               "UNDUP_CT_PARTIC_MEN", "UNDUP_CT_PARTIC_WOMEN", "RECRUITEXP_TOTAL",
               "GRND_TOTAL_EXPENSE", "EFTotalCount"]
needed_sports = ["unitid", "Sports", "PARTIC_MEN", "PARTIC_WOMEN", "PARTIC_COED_MEN", "PARTIC_COED_WOMEN"]
print("Missing inst cols:", [c for c in needed_inst if c not in inst.columns])
print("Missing sports cols:", [c for c in needed_sports if c not in sports.columns])

Scorecard: (3147, 31) | EADA schools: (2037, 168) | EADA sport rows: (18045, 129)
Missing inst cols: []
Missing sports cols: []


In [2]:
# Test the "no athletic scholarships" rule with data instead of assuming it.
# If D-III, NJCAA D-III, and CCCAA are right, % giving aid should be ~0.
check = inst.groupby("classification_name").agg(
    schools=("unitid", "count"),
    pct_giving_aid=("STUDENTAID_TOTAL", lambda s: (s.fillna(0) > 0).mean() * 100),
    median_aid=("STUDENTAID_TOTAL", "median"),
).round(0).sort_values("median_aid")
print(check.to_string())

                                    schools  pct_giving_aid  median_aid
classification_name                                                    
CCCAA                                   113             3.0         0.0
NJCAA Division III                       92            16.0         0.0
NCCAA Division II                        29             3.0         0.0
NCAA Division III without football      169             4.0         0.0
NCAA Division III with football         235             5.0         0.0
Independent                               9            44.0         0.0
USCAA                                    47            26.0         0.0
NJCAA Division II                       157            98.0    173564.0
NWAC                                     32           100.0    197202.0
Other                                    62            85.0    218371.0
NCCAA Division I                          9            89.0    329500.0
NJCAA Division I                        206            97.0    6

In [3]:
# Group messy sport names so e.g. a track athlete matches every track-type team.
sport_groups = {
    "Track and Field and Cross Country (combined)": "Track & Field / Cross Country",
    "Track and Field (Outdoor)": "Track & Field / Cross Country",
    "Track and Field (Indoor)": "Track & Field / Cross Country",
    "Cross Country": "Track & Field / Cross Country",
    "Swimming and Diving (combined)": "Swimming & Diving",
    "Swimming": "Swimming & Diving",
    "Diving": "Swimming & Diving",
}
sports["sport"] = sports["Sports"].replace(sport_groups)

# A team counts for men if it has any male participants (men's team or coed); same for women.
sports["has_men"] = (sports["PARTIC_MEN"].fillna(0) + sports["PARTIC_COED_MEN"].fillna(0)) > 0
sports["has_women"] = (sports["PARTIC_WOMEN"].fillna(0) + sports["PARTIC_COED_WOMEN"].fillna(0)) > 0

def join_list(s):
    return "; ".join(sorted(set(s)))   # e.g. "Baseball; Basketball; Soccer"

mens = sports[sports["has_men"]].groupby("unitid")["sport"].agg(join_list).rename("mens_sports").reset_index()
womens = sports[sports["has_women"]].groupby("unitid")["sport"].agg(join_list).rename("womens_sports").reset_index()
print("Schools with men's teams:", len(mens), "| with women's teams:", len(womens))
print("Example:", mens.iloc[0].to_dict())

Schools with men's teams: 2027 | with women's teams: 2024
Example: {'unitid': 100654, 'mens_sports': 'Baseball; Basketball; Football; Golf; Tennis; Track & Field / Cross Country'}


In [4]:
# School-level athletics features, then LEFT JOIN onto our schools (keeps all 3,147).
ath = inst[needed_inst].rename(columns={
    "classification_name": "division",
    "STUDENTAID_MEN": "athletic_aid_men",
    "STUDENTAID_WOMEN": "athletic_aid_women",
    "STUDENTAID_TOTAL": "athletic_aid_total",
    "UNDUP_CT_PARTIC_MEN": "athletes_men",        # unduplicated: a 2-sport athlete counts once
    "UNDUP_CT_PARTIC_WOMEN": "athletes_women",
    "RECRUITEXP_TOTAL": "recruiting_spend",
    "GRND_TOTAL_EXPENSE": "athletics_expense",
    "EFTotalCount": "eada_enrollment",
})
# Per-athlete aid = rough sense of how much scholarship money exists per roster spot
ath["aid_per_athlete_men"] = ath["athletic_aid_men"] / ath["athletes_men"].replace(0, np.nan)
ath["aid_per_athlete_women"] = ath["athletic_aid_women"] / ath["athletes_women"].replace(0, np.nan)
# Spending per student = candidate proxy for "sport culture" (we'll evaluate it next step)
ath["athletics_spend_per_student"] = ath["athletics_expense"] / ath["eada_enrollment"].replace(0, np.nan)

ath = ath.merge(mens, on="unitid", how="left").merge(womens, on="unitid", how="left")
print("Duplicate unitids in EADA (should be 0):", ath["unitid"].duplicated().sum())

merged = schools.merge(ath, left_on="unit_id", right_on="unitid", how="left", indicator=True)
merged["has_athletics"] = merged["_merge"] == "both"
merged = merged.drop(columns=["_merge", "unitid"])
assert len(merged) == len(schools), "Row count changed: merge created duplicates"

print("\nEADA schools NOT in our universe (for-profits, etc.):", (~inst["unitid"].isin(schools["unit_id"])).sum())
print("\nHas athletics, by school type:")
print(pd.crosstab(merged["school_type"], merged["has_athletics"], margins=True))
print("\nDivisions within our universe:")
print(merged["division"].value_counts(dropna=False).to_string())

merged.to_csv("../data/processed/schools_with_athletics.csv", index=False)
print("\nSaved:", merged.shape, "-> data/processed/schools_with_athletics.csv")

Duplicate unitids in EADA (should be 0): 0

EADA schools NOT in our universe (for-profits, etc.): 15

Has athletics, by school type:
has_athletics  False  True   All
school_type                     
2-year           666   684  1350
4-year           459  1338  1797
All             1125  2022  3147

Divisions within our universe:
division
<NA>                                  1125
NCAA Division III with football        235
NJCAA Division I                       203
NCAA Division III without football     169
NAIA Division I                        164
NJCAA Division II                      157
NCAA Division II with football         155
NCAA Division II without football      136
NCAA Division I-FCS                    131
NCAA Division I-FBS                    128
CCCAA                                  113
NCAA Division I without football        99
NJCAA Division III                      92
Other                                   59
NAIA Division II                        59
USCAA           

In [5]:
df = merged.copy()

# 1. Validation: which FCS schools reported $0 athletic aid? (Expect the Ivy League.)
fcs_zero = inst[(inst["classification_name"] == "NCAA Division I-FCS") & (inst["STUDENTAID_TOTAL"].fillna(0) == 0)]
print("FCS schools with $0 athletic aid:")
print(fcs_zero["institution_name"].to_string(index=False))

# 2. Pull in EADA's free-text "Other" description for each school
df["division_other"] = df["unit_id"].map(inst.set_index("unitid")["ClassificationOther"])

# 3. Simplified association group. "Other" schools are remapped using their free-text description.
def association(d, other):
    if pd.isna(d):
        return None
    if d == "Other":
        o = str(other).upper()
        if any(k in o for k in ["NJCAA", "NJCCA", "NWAC", "JUCO", "JUNIOR COLLEGE"]):
            return "JUCO"
        if any(k in o for k in ["LAI", "LIGA", "PUERTO RICO", "INTERUNIVERSIT", "INTER-UNIVERSIT"]):
            return "Puerto Rico (LAI)"
        return "Other small-college"
    if d.startswith("NCAA Division I") and not d.startswith("NCAA Division II"): return "NCAA D1"
    if d.startswith("NCAA Division III"): return "NCAA D3"
    if d.startswith("NCAA Division II"): return "NCAA D2"
    if d.startswith("NAIA"): return "NAIA"
    if d.startswith("NJCAA") or d in ["CCCAA", "NWAC"]: return "JUCO"
    return "Other small-college"

df["association"] = df.apply(lambda r: association(r["division"], r["division_other"]), axis=1)

# Human review: every "Other" description and where it was mapped
print("How 'Other' schools were remapped:")
print(df[df["division"] == "Other"].groupby(["association", "division_other"]).size().to_string())

# 4a. Athletic scholarship tier: division rule (tested in cell 2) + the school's own reported aid
no_aid_divisions = ["CCCAA", "NJCAA Division III", "NCCAA Division II",
                    "NCAA Division III without football", "NCAA Division III with football"]
mixed_divisions = ["Independent", "USCAA", "Other"]

def aid_tier(row):
    if not row["has_athletics"]:
        return "No athletics"
    if row["division"] in no_aid_divisions or row["athletic_aid_total"] == 0:
        return "No athletic scholarships"
    if row["division"] in mixed_divisions:
        # Multi-division JUCOs that report aid do give athletic scholarships
        return "Athletic scholarships" if row["association"] == "JUCO" else "Mixed / verify"
    return "Athletic scholarships"

df["athletic_aid_tier"] = df.apply(aid_tier, axis=1)
print("\nAthletic scholarship tier:")
print(pd.crosstab(df["athletic_aid_tier"], df["school_type"], margins=True))
print("\nAssociation counts:")
print(df["association"].value_counts(dropna=False).to_string())
print("\nNeosho check:")
print(df[df["name"].str.contains("Neosho", na=False)][["name", "division", "division_other", "association", "athletic_aid_tier"]].to_string(index=False))

# 4. Two candidate "sport culture" measures (comparison only)
df["athlete_share"] = (df["athletes_men"].fillna(0) + df["athletes_women"].fillna(0)) / df["eada_enrollment"]
print("\nMedian by association (only schools with athletics):")
print(df[df["has_athletics"]].groupby("association").agg(
    schools=("unit_id", "count"),
    median_total_spend=("athletics_expense", "median"),        # (a) big-time sports
    median_spend_per_student=("athletics_spend_per_student", "median"),
    median_athlete_share=("athlete_share", "median"),          # (b) sports central to campus
).round(2).to_string())

cols = ["name", "state", "association", "undergrads", "athletics_expense", "athlete_share"]
print("\nTop 10 by TOTAL athletics spending (definition a):")
print(df.nlargest(10, "athletics_expense")[cols].round(2).to_string(index=False))
print("\nTop 10 by ATHLETE SHARE, 1,000+ undergrads (definition b):")
print(df[df["undergrads"] >= 1000].nlargest(10, "athlete_share")[cols].round(2).to_string(index=False))

# 5. Sport culture = how big and well-known the athletics program is (definition a).
#    Proxy: total athletics spending as a percentile among schools WITH athletics
#    (0 = smallest program, 100 = biggest). Schools with no athletics get 0.
df["sport_culture_pct"] = (df["athletics_expense"].rank(pct=True) * 100).round(1)
df.loc[~df["has_athletics"], "sport_culture_pct"] = 0
print("\nSport culture percentile, median by association:")
print(df.groupby("association")["sport_culture_pct"].median().sort_values(ascending=False))

df.to_csv("../data/processed/schools_with_athletics.csv", index=False)
print("\nSaved:", df.shape)

FCS schools with $0 athletic aid:
                            Yale University
                         Harvard University
                          Dartmouth College
                       Princeton University
Columbia University in the City of New York
                         Cornell University
                 University of Pennsylvania
                           Brown University
How 'Other' schools were remapped:
association          division_other                                    
JUCO                 NJCAA                                                 1
                     NJCAA - Multiple Divisions Based On Sports            1
                     NJCAA D1 and NJCAA D2                                 1
                     NJCAA D2 - Lax Non-divisional                         1
                     NJCAA DI and DII depending on sport                   1
                     NJCAA DI and DII, ACHA DII                            1
                     NJCAA Div I and II      